# FLUX.2 Klein-4B on TPU

Runner for [JohanesSetiawan/flux2-tpu](https://github.com/JohanesSetiawan/flux2-tpu).

This notebook contains no logic. Every cell is a call into the `src`
package; the work happens there, where it is version controlled and
tested. If you find yourself writing an algorithm in a cell here, it
belongs in the package instead.

**Before running:** select a TPU accelerator. On Kaggle that is under
Settings, Accelerator. The pipeline runs on CPU too, but slowly enough
that it is only useful for checking the wiring.

**No safety filtering.** This codebase includes no content moderation
or output filtering of any kind. You are responsible for what you
generate with it.

## 1. Fetch the code

In [ ]:
!git clone --depth 1 https://github.com/JohanesSetiawan/flux2-tpu.git
%cd flux2-tpu

## 2. Install dependencies

`jax[tpu]` is installed only when a TPU is present. On Kaggle the TPU
runtime usually ships a working JAX already, so this upgrades rather
than installs from nothing.

In [ ]:
%pip install --quiet -r requirements.txt
%pip install --quiet ipywidgets gradio pillow

## 3. Configure

`compilation_cache_directory` is worth setting on a hosted notebook.
Compiled programs then survive a session restart, which otherwise costs
a full recompilation every time the session is recycled.

In [ ]:
import logging
from pathlib import Path

from src.config import (
    CheckpointSourceConfig,
    ExecutionConfig,
    InferenceConfig,
    MemoryResidencyStrategy,
)
from src.utils import configure_logging

WORKING_DIRECTORY = Path("/kaggle/working")

logger = configure_logging(
    log_file_path=WORKING_DIRECTORY / "generation_log.txt",
    logger_name="flux2_klein",
)

inference_config = InferenceConfig(
    checkpoint_source=CheckpointSourceConfig(
        local_cache_directory=Path("/kaggle/temp/flux2_klein_checkpoint_cache"),
    ),
    # AUTO picks a residency strategy from the visible device count:
    # a single chip holds the transformer and decoder and swaps the text
    # encoder, a pod holds everything. Override only if you know why.
    residency_strategy=MemoryResidencyStrategy.AUTO,
)

execution_config = ExecutionConfig(
    compilation_cache_directory=WORKING_DIRECTORY / "compilation_cache",
)

## 4. Load

Downloads roughly 13 GB the first time and restores three components.
Expect several minutes; later runs reuse the local cache.

In [ ]:
from src.pipeline import Pipeline

pipeline = Pipeline(inference_config, logger, execution_config=execution_config)
pipeline.load()

## 5. Warm up

Compilation is per output shape, not per prompt, so this pays the
compile cost once for each resolution rather than on the first request
that uses it. Skip it if you only want one image and do not care
whether the wait comes now or in a moment.

In [ ]:
pipeline.warm_up()

## 6. Generate

Two interfaces over the same pipeline. Run whichever you prefer; they
share all their input handling, so they behave identically.

A seed of `-1` draws a random one. The seed that was actually used is
reported after each generation, so a result you like can be reproduced.

### Option A: in-notebook controls

In [ ]:
from IPython.display import display

from src.interfaces.widgets import build_control_panel

display(build_control_panel(pipeline, logger))

### Option B: browser interface

`share=True` creates a public link, which is how a hosted notebook is
usually reached. That link is public while the cell runs; leave it off
if you would rather not expose the interface.

In [ ]:
from src.interfaces.browser import build_interface

build_interface(pipeline, logger).launch(share=True)

### Option C: call it directly

Useful for scripting a batch, and the shortest path to a single image.

In [ ]:
from src.interfaces.session import build_request, to_display_image
from PIL import Image

request = build_request(
    prompt="a lighthouse on a rocky shore at dusk",
    resolution_label="1024x1024",
    requested_seed=-1,
    buckets=pipeline.resolution_buckets,
)

image = pipeline.generate(request)
print(f"seed {request.seed}")
Image.fromarray(to_display_image(image))

## Notes

Repeated prompts skip the text encoder: conditioning is cached by
prompt text, so changing only the seed or resolution reuses it. Under
the swapped residency strategy that also avoids moving the encoder back
into accelerator memory, which is the dominant cost of a repeat.

Only three resolutions are offered, and that is deliberate rather than
a limitation of the interface. Above roughly 4300 image tokens the
reference sampling schedule switches to a formula derived for a
two-hundred-step model, which this four-step checkpoint was not tuned
against. See AGENTS.md for the measured discontinuity.

The full log is written to `generation_log.txt` in the working
directory, and survives the notebook output being cleared.